In [22]:
from pymongo import MongoClient

db= MongoClient("mongodb://localhost:27017")["panteon"]

## Consulta 5
¿Cuál es la longitud promedio, mínima y máxima de los TEs por clase, orden y superfamilia?

In [23]:
def longitud_por(campo):
    print(f"{campo:<20}{'total':>8}{'prom':>10}{'min':>7}{'max':>8}")
    for d in db.secuencias.aggregate([
        {"$group":{"_id":"$"+campo,"n":{"$sum":1},"prom":{"$avg":"$longitud"},"min":{"$min":"$longitud"},"max":{"$max":"$longitud"}}},
        {"$sort":{"_id":1}}
    ]):
        print(f"{d['_id']:<20}{d['n']:>8}{d['prom']:>10.1f}{d['min']:>7}{d['max']:>8}")

longitud_por("te_class")
longitud_por("te_order")
longitud_por("te_superfamily")

te_class               total      prom    min     max
CLASSI                  1367    4008.7     97   13785
CLASSII                  633    1258.9    106   18485
te_order               total      prom    min     max
CRYPTON                    3     838.0    305    1725
DIRS                      12    2874.4    354    6632
HELITRON                  89    1154.1    159   10898
LINE                     607    3378.1    510    5872
LTR                      698    4841.7    303   13785
MAVERICK                   2   14788.5  11092   18485
MITE                     173     349.6    106     500
PLE                        1    3728.0   3728    3728
RETROPOSON                 1    1212.0   1212    1212
SINE                      48     220.0     97     358
TIR                      366    1643.7    128    9968
te_superfamily         total      prom    min     max
ALU                        1     320.0    320     320
BELPAO                    45    7256.7   3140    8252
CMC                       32

se puede ver que la clase i tiene más secuencias que la clase ii, además su longitud promedio es bastante mayor, con unas 4008 bases frente a unas 1259, aunque en la clase ii aparece la secuencia más larga de todas, con 18485 bases

por orden, los ltr tienen una longitud promedio mayor que la mayoría de los demás, con unas 4842 bases, mientras que los sine tienen las secuencias más cortas, con un promedio de 220 bases, también se ve que line y tir tienen bastantes secuencias en comparación con otros órdenes, mientras que algunos como ple y retroposon tienen muy pocos registros

por superfamilia hay bastante variedad en las longitudes, algunas como belpao y copia tienen promedios altos, mientras que otras como trna tienen secuencias mucho más cortas, también hay superfamilias con muy pocos registros, así que sus promedios no representan tantos datos como los de las que tienen más secuencias, en general se puede ver que la longitud cambia bastante dependiendo de la clase, el orden y la superfamilia del te

## Consulta 6
¿Cuál es el porcentaje de GC promedio por orden y por reino?

In [24]:
print(f"{'orden':<20}{'total':>8}{'gc':>8}")
for d in db.secuencias.aggregate([
    {"$group":{"_id":"$te_order","n":{"$sum":1},"gc":{"$avg":"$porcentaje_gc"}}},
    {"$sort":{"gc":-1}}
]):
    print(f"{d['_id']:<20}{d['n']:>8}{d['gc']:>8.2f}")

print(f"{'reino':<20}{'total':>8}{'gc':>8}")
for d in db.secuencias.aggregate([
    {"$project":{"especie_id":1,"porcentaje_gc":1}},
    {"$lookup":{"from":"especies","localField":"especie_id","foreignField":"_id","as":"sp"}},
    {"$unwind":"$sp"},
    {"$group":{"_id":"$sp.reino","n":{"$sum":1},"gc":{"$avg":"$porcentaje_gc"}}},
    {"$sort":{"gc":-1}}
]):
    print(f"{str(d['_id']):<20}{d['n']:>8}{d['gc']:>8.2f}")

orden                  total      gc
PLE                        1   60.93
HELITRON                  89   60.13
SINE                      48   55.96
CRYPTON                    3   52.92
RETROPOSON                 1   49.59
DIRS                      12   49.35
LTR                      698   48.68
LINE                     607   41.67
TIR                      366   39.86
MITE                     173   39.80
MAVERICK                   2   35.57
reino                  total      gc
Animalia                2000   44.86


se puede ver que el porcentaje de gc cambia bastante dependiendo del orden, los ple tienen el promedio más alto con un 60.93%, aunque solo hay un registro, algo parecido pasa con los helitron, que tienen un 60.13% y 89 secuencias, mientras que los sine también tienen un porcentaje relativamente alto con un 55.96%

por otro lado, los line tienen un promedio de 41.67% y los tir y mite están cerca del 40%, los maverick tienen el porcentaje más bajo con un 35.57%, aunque solo hay dos registros, en general se observa que algunos órdenes tienen una mayor proporción de guanina y citosina que otros

por reino, solo aparece animalia en los resultados, con 2000 secuencias y un promedio de gc de 44.86%, así que con estos datos no se puede hacer una comparación entre distintos reinos, ya que no hay registros de otros reinos en el resultado

## Consulta 7
¿Qué TEs son los más largos y los más cortos de cada reino? Muestren su nombre, clasificación y especie.

In [25]:
def extremo_por_reino(sentido):
    for d in db.secuencias.aggregate([
        {"$project":{"secuencia":0}},
        {"$lookup":{"from":"especies","localField":"especie_id","foreignField":"_id","as":"sp"}},
        {"$unwind":"$sp"},
        {"$sort":{"longitud":sentido,"id_original":1}},
        {"$group":{"_id":"$sp.reino","id":{"$first":"$id_original"},"clase":{"$first":"$te_class"},"orden":{"$first":"$te_order"},"sf":{"$first":"$te_superfamily"},"especie":{"$first":"$especie"},"longitud":{"$first":"$longitud"}}},
        {"$sort":{"_id":1}}
    ],allowDiskUse=True):
        print(f"{str(d['_id']):<10}{d['id']:<16}{d['clase']:<9}{d['orden']:<12}{d['sf']:<16}{d['longitud']:>7}  {d['especie']}")

print("mas largos")
extremo_por_reino(-1)
print("mas cortos")
extremo_por_reino(1)

mas largos
Animalia  Polinton-1_DR   CLASSII  MAVERICK    MAVERICK          18485  Danio rerio
mas cortos
Animalia  GalVar-1.2      CLASSI   SINE        TRNA                 97  Galeopterus variegatus


en el reino animalia, el te más largo es polinton-1_dr, pertenece a la clase ii, al orden maverick y tiene una longitud de 18485 bases, se encontró en la especie danio rerio, por otro lado, el te más corto es galvar-1.2, pertenece a la clase i, al orden sine y a la superfamilia trna, tiene una longitud de apenas 97 bases y se encontró en la especie galeopterus variegatus, esto muestra que hay una diferencia bastante grande entre las longitudes de los elementos transponibles encontrados en los animales, desde secuencias muy cortas hasta otras mucho más largas

## Consulta 8
Buscar una secuencia por su nombre (por ejemplo, PDB00000001) y mostrar su clasificación, especie y taxonomía completa.

In [26]:
def buscar(nombre):
    seq= db.secuencias.find_one({"id_original":nombre},{"secuencia":0})
    if seq is None:
        print("no existe")
        return
    sp= db.especies.find_one({"_id":seq["especie_id"]})
    print("secuencia:",seq["id_original"])
    print("clasificacion:",seq["te_class"],seq["te_order"],seq["te_superfamily"])
    print("especie:",sp["especie"])
    print("taxonomia:",sp["reino"],sp["filo"],sp["tax_clase"],sp["tax_orden"],sp["familia"],sp["especie"])

buscar("PDB00000001")

secuencia: PDB00000001
clasificacion: CLASSII HELITRON HELITRON
especie: Halicephalobus mephisto
taxonomia: Animalia Nematoda Chromadorea Rhabditida Panagrolaimidae Halicephalobus mephisto


al buscar la secuencia PDB00000001, se encontró que pertenece a la clase ii, al orden helitron y a la superfamilia helitron, la especie en la que se encuentra es halicephalobus mephisto, que pertenece al reino animalia, al filo nematoda, a la clase chromadorea, al orden rhabditida y a la familia panagrolaimidae, con esto se puede ver que la consulta permite encontrar una secuencia por su nombre y conocer tanto su clasificación como la información taxonómica completa de la especie a la que pertenece

## Consulta 9
Listar los nombres de todas las secuencias de una superfamilia en una especie dada.

In [27]:
def secuencias_de(especie,superfamilia):
    total= 0
    for d in db.secuencias.find({"especie":especie,"te_superfamily":superfamilia.upper()},{"id_original":1,"_id":0}).sort("id_original",1):
        print(d["id_original"])
        total+= 1
    print("total:",total)

secuencias_de("Halicephalobus mephisto","HELITRON")

PDB00000001
PDB00000002
total: 2


en la especie halicephalobus mephisto se encontraron dos secuencias pertenecientes a la superfamilia helitron, identificadas como PDB00000001 y PDB00000002, la consulta permite listar los nombres de las secuencias que pertenecen a una superfamilia específica dentro de una especie y también muestra el total de coincidencias encontradas